# Python Master Notes — Time, Decorators, Modules & Packages
**Class date: 24 October 2026**

This notebook combines the supplied class notebook, Python scripts, package examples and class transcript into one structured set of notes. It follows the order of the session:

1. The `time` module and measuring execution time
2. Decorators: function objects, wrappers, arguments, stacking and practical transformations
3. Modules: separating reusable code into `.py` files
4. Packages: organizing modules into folders and exposing selected names through `__init__.py`
5. `__name__ == "__main__"` and safe module testing

> The supplied PDF did not contain extractable text. The readable notebook, scripts and transcript are the basis for these notes. The original examples are preserved in meaning, with technical clarifications called out explicitly where necessary.

## How to use this notebook
- Run the self-contained examples from top to bottom.
- For module/package exercises, create the folder and files shown in the examples. A notebook cell cannot fully replace a real multi-file project.
- Cells that ask for interactive input are commented out or separated so the notebook does not unexpectedly block.
- Use the dry runs to trace which function is called and what each variable refers to.

# Part 1 — Python `time` module

## 1.1 Importing the module
`time` is part of Python's standard library. It does not need a separate `pip install`.

In [ ]:
import time

print(time.time())
print(time.perf_counter())

## 1.2 `time.time()` — wall-clock timestamp
`time.time()` returns the current time as a floating-point number of seconds since the Unix epoch: 1 January 1970 UTC.

Use it when you need a wall-clock timestamp, for example, to record when an event happened.

**Precision clarification:** the value is expressed in seconds, not milliseconds. The fractional part may represent sub-second precision, but actual clock resolution depends on the system.

In [ ]:
t1 = time.time()
# Perform some work or wait for a short interval.
time.sleep(1)
t2 = time.time()

elapsed_seconds = t2 - t1
print(f"Elapsed wall-clock time: {elapsed_seconds:.3f} seconds")

### Dry run
1. `t1` records the wall-clock timestamp before the wait.
2. `time.sleep(1)` asks the current thread to pause for approximately one second.
3. `t2` records a later wall-clock timestamp.
4. `t2 - t1` gives the elapsed number of seconds between the two timestamps.

If you rerun only the subtraction cell later, it uses the old values of `t1` and `t2`; rerun the whole measurement block to take a new measurement.

## 1.3 `time.perf_counter()` — measuring durations
`time.perf_counter()` is designed for measuring short durations. It uses a high-resolution performance counter. Its reference point is undefined, so the absolute number is not normally meaningful; subtract two readings to measure elapsed time.

Use it for timing code execution, API-call duration measurements and performance comparisons.

In [ ]:
start = time.perf_counter()

# Example workload
total = 0
for number in range(100_000):
    total += number

end = time.perf_counter()
print(f"Workload took {end - start:.6f} seconds")

### `time.time()` versus `time.perf_counter()`
| Function | What it measures | Typical use |
|---|---|---|
| `time.time()` | Current wall-clock timestamp (seconds since Unix epoch) | Event timestamps / calendar time |
| `time.perf_counter()` | High-resolution counter with an undefined reference point | Elapsed duration / benchmarking |
| `time.sleep(seconds)` | Suspends the current thread for approximately the requested duration | Waiting / pacing |

The transcript describes `perf_counter()` as the higher-precision option. More precisely, it is the recommended high-resolution clock for duration measurement. Do not compare its absolute reading to a calendar timestamp.

## 1.4 `time.sleep()`
`time.sleep(seconds)` pauses the current thread for at least approximately the requested duration. The actual delay can be longer because of operating-system scheduling and other activity.

In [ ]:
print("Before sleep")
time.sleep(2)
print("After approximately two seconds")

## 1.5 Measure a loop
The original notebook used a loop with `...` in its body. The ellipsis literal is a valid Python expression and can act as a placeholder when no operation is required.

In [ ]:
def loop_trial(iterations=1_000_000):
    start = time.perf_counter()

    for _ in range(iterations):
        ...

    end = time.perf_counter()
    return end - start

print(f"Loop duration: {loop_trial():.6f} seconds")

**Performance tip:** keep the workload identical when comparing two implementations. Run measurements more than once and compare several runs; one measurement can be affected by background activity, caching and scheduling.

# Part 2 — Decorators

## 2.1 Why decorators?
Suppose a calculation function already works, but you want to add a greeting before it runs and a closing message after it runs. You could write those statements around every call, but that duplicates the enhancement at every call site.

A decorator lets you add behaviour around a function without editing the function's original implementation.

**Class definition:** a decorator is a function that takes another function as an argument and returns a new function (or another callable) that enhances or changes its behaviour.

## 2.2 Functions are objects
A function can be:
- assigned to a variable,
- passed as an argument to another function,
- returned from another function.

Writing `print_hello` refers to the function object. Writing `print_hello()` calls the function.

In [ ]:
def print_hello():
    print("Hello " * 3)

print(type(print_hello))
print(id(print_hello))

# Passing the function object, not calling it yet:
def run(func):
    func()

run(print_hello)

### Dry run
- `run(print_hello)` passes the function object into the parameter named `func`.
- Inside `run`, `func` refers to `print_hello`.
- `func()` calls the function.
- If you wrote `run(print_hello())`, the function would run immediately and its return value (here `None`) would be passed instead.

## 2.3 A function can return another function
A nested function can use values from its enclosing function. Returning the nested function gives the caller a new callable that remembers the original function reference.

In [ ]:
def run_with_messages(func):
    def wrapper():
        print("Start of the function")
        func()
        print("End of the function")
    return wrapper

enhanced_print = run_with_messages(print_hello)
enhanced_print()

What happened?
1. `run_with_messages(print_hello)` passes the function object to the outer function.
2. The outer function defines `wrapper`.
3. It returns the `wrapper` function object. It does not call it yet.
4. `enhanced_print` now refers to the returned wrapper.
5. `enhanced_print()` calls the wrapper, which prints the start message, calls the original function and prints the end message.

## 2.4 First decorator example — calculation greeting
The class notebook used a calculation function that accepts a calculation name and asks the user for two numbers. This version uses return values so the behaviour is easy to test without interactive input.

In [ ]:
def calculation(calculation_name, value_1, value_2):
    if calculation_name == "add":
        return value_1 + value_2
    elif calculation_name == "subtract":
        return value_1 - value_2
    return None

def add_greetings(func):
    def wrapper(calculation_name, value_1, value_2):
        print("Hello")
        result = func(calculation_name, value_1, value_2)
        print("Thanks for using our function.")
        return result
    return wrapper

calculation = add_greetings(calculation)

print(calculation("add", 12, 3))

The original function name `calculation` is rebound to the returned wrapper. Calling `calculation(...)` now calls the wrapper, and the wrapper calls the original function through `func`.

This is the same transformation written with decorator syntax:

In [ ]:
def add_greetings(func):
    def wrapper(calculation_name, value_1, value_2):
        print("Hello")
        result = func(calculation_name, value_1, value_2)
        print("Thanks for using our function.")
        return result
    return wrapper

@add_greetings
def calculation(calculation_name, value_1, value_2):
    if calculation_name == "add":
        return value_1 + value_2
    elif calculation_name == "subtract":
        return value_1 - value_2
    return None

print(calculation("subtract", 20, 7))

`@add_greetings` is syntactic sugar for:

```python
def calculation(...):
    ...

calculation = add_greetings(calculation)
```

The decorator is applied when the function definition is executed, not every time the decorated function is called.

## 2.5 The wrapper must accept the arguments
A common beginner issue: the decorated function accepts arguments, but the wrapper accepts none. After decoration, the name points to the wrapper, so calling it with arguments raises `TypeError`.

Use `*args` and `**kwargs` when you want a general-purpose wrapper that can forward positional and keyword arguments.

In [ ]:
def trace_call(func):
    def wrapper(*args, **kwargs):
        print("Starting function")
        result = func(*args, **kwargs)
        print("Function finished")
        return result
    return wrapper

@trace_call
def add_numbers(a, b, c=0):
    return a + b + c

print(add_numbers(10, 20))
print(add_numbers(10, 20, c=30))

### `args` and `kwargs` inside the wrapper
- `args` is a tuple of positional arguments.
- `kwargs` is a dictionary of keyword arguments.
- `func(*args, **kwargs)` unpacks them and forwards them to the original function.
- Returning `result` preserves the original function's return value.

A wrapper should normally return the wrapped function's result. Otherwise, a function that originally returned a useful value may appear to return `None` after decoration.

## 2.6 Decorator around a flexible addition function
The supplied class notebook used `*args` and `**kwargs` to accept many values. This cleaned-up version accepts numeric positional and keyword values and ignores non-numeric keyword values, matching the original teaching intention.

In [ ]:
def add_values(*args, **kwargs):
    total = 0

    for value in args:
        total += float(value)

    for key, value in kwargs.items():
        try:
            total += float(value)
        except (TypeError, ValueError):
            # Non-numeric keyword values are skipped in this example.
            pass

    return total

def decorator_with_arguments(func):
    def wrapper(*args, **kwargs):
        print("Hello, please wait ...")
        result = func(*args, **kwargs)
        print("Executed.")
        return result
    return wrapper

add_values = decorator_with_arguments(add_values)
print(add_values(1, 2, c=100, b=200, label="demo"))

**Terminology clarification:** `decorator_with_arguments` here accepts the *function* as its argument. The returned `wrapper` accepts the arguments that will later be passed to the decorated function. These are two different calls at two different stages.

## 2.7 Decorator call flow — dry run
For:

```python
@decorator_with_arguments
def add_values(*args, **kwargs):
    ...
```

Python effectively performs:

```python
def add_values(*args, **kwargs):
    ...

add_values = decorator_with_arguments(add_values)
```

Step by step:
1. Python creates the original function object named `add_values`.
2. Python passes that object to `decorator_with_arguments`.
3. The decorator defines `wrapper`, where `func` refers to the original `add_values`.
4. The decorator returns `wrapper`.
5. The name `add_values` is rebound to the returned wrapper.
6. When `add_values(1, 2, c=100)` is called, the wrapper receives those values in its `args` and `kwargs`.
7. The wrapper calls the original function using `func(*args, **kwargs)`.

## 2.8 Multiple decorators
A function can have more than one decorator. The decorator closest to the function is applied first; the outer decorator is applied to the result.

In [ ]:
def dec_1(func):
    def wrap():
        print("Inside decorator 1 — before")
        func()
        print("Inside decorator 1 — after")
    return wrap

def dec_2(func):
    def wrap():
        print("Inside decorator 2 — before")
        func()
        print("Inside decorator 2 — after")
    return wrap

@dec_2
@dec_1
def show_message():
    print("Original function")

show_message()

The syntax above is equivalent to:

```python
show_message = dec_2(dec_1(show_message))
```

So `dec_1` wraps the original function first, and `dec_2` wraps the result of `dec_1`.

## 2.9 Practical example — strip spaces and uppercase
The original notebook defined two decorators:
- `strip_spaces`: removes leading and trailing whitespace.
- `make_upper_case`: converts the value to uppercase.

The original function returns the transformed greeting.

In [ ]:
def make_upper_case(func):
    def wrap(value):
        value = value.upper()
        return func(value)
    return wrap

def strip_spaces(func):
    def wrap(value):
        value = value.strip()
        return func(value)
    return wrap

@make_upper_case
@strip_spaces
def greet(greeting):
    print(greeting)
    return greeting

result = greet("   hello, how are you   ")
print("Returned:", repr(result))

### Transformation order
The decorators are applied as:

```python
greet = make_upper_case(strip_spaces(greet))
```

When called:
1. The outer `make_upper_case` wrapper runs first and uppercases the original input.
2. It passes the result to the inner `strip_spaces` wrapper.
3. The inner wrapper strips leading and trailing spaces.
4. The original `greet` function receives the final value.

The final string is uppercase and has no leading/trailing spaces.

## 2.10 A production-friendly decorator: preserve metadata
A standard decorator often uses `functools.wraps` so the wrapper retains the original function's name, docstring and metadata.

In [ ]:
from functools import wraps

def log_call(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Calling {func.__name__}")
        return func(*args, **kwargs)
    return wrapper

@log_call
def multiply(a, b):
    """Multiply two values and return the result."""
    return a * b

print(multiply(6, 7))
print(multiply.__name__)
print(multiply.__doc__)

### Decorator revision checklist
- A decorator receives a callable.
- It usually defines and returns a wrapper.
- The wrapper calls the original callable.
- `*args, **kwargs` allow general argument forwarding.
- Return the original result if you want to preserve return behaviour.
- `@decorator` is shorthand for rebinding the function name to the decorator's return value.
- Multiple decorators are applied from the closest-to-function decorator outward.
- `functools.wraps` preserves useful metadata.

# Part 3 — Modules

## 3.1 What is a module?
A **module** is a Python file (`.py`) that contains Python code. It can contain functions, classes and variables that can be imported and reused from another file.

The class used `healthcare.py` to keep healthcare-related reusable functions separate from `main.py`.

## 3.2 Example project structure
Create a folder and place these two files in it:

```text
healthcare_demo/
    healthcare.py
    main.py
```

**healthcare.py**

In [ ]:
# Save this code in healthcare.py

def calculate_bmi(weight_kg, height_meters):
    if weight_kg <= 0 or height_meters <= 0:
        return "Error: Weight and height must be greater than zero."

    bmi = round(weight_kg / (height_meters ** 2), 1)

    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25:
        category = "Normal weight"
    elif bmi < 30:
        category = "Overweight"
    else:
        category = "Obesity"

    return {"bmi": bmi, "category": category}

**main.py**

In [ ]:
# Save this code in main.py, in the same folder as healthcare.py

import healthcare

person_a_bmi = healthcare.calculate_bmi(70, 1.75)
print(person_a_bmi)

Run the script from the project folder:

```bash
python main.py
```

Python imports the module, then `healthcare.calculate_bmi(...)` calls the function defined in that module. The main file does not need to contain the function's implementation.

## 3.3 Import styles

In [ ]:
# Import the module, then use module_name.member_name
import math
print(math.sqrt(81))

# Import selected names from a module
from math import sqrt, pi
print(sqrt(100))
print(pi)

# Import a module using an alias
import math as m
print(m.factorial(5))

The class demonstrated importing an entire module, importing selected functions, importing multiple selected names and using `*` to import public names.

`from module import *` is usually discouraged in larger projects because it can make it unclear where a name came from and can create naming conflicts. Prefer explicit imports.

## 3.4 A reusable calculator module
The supplied `calculator.py` contains a calculator class that stores addition history. The original class name is misspelled as `Calulcator`; below it is written as `Calculator` for clarity.

In [ ]:
# Save this code in calculator.py

import time

class Calculator:
    def __init__(self):
        self.__history = []

    def add(self, *args):
        total = sum(args)
        timestamp = time.time()
        self.__history.append({
            f"Add_{timestamp}": {str(args): total}
        })
        return total

    def get_history(self):
        return self.__history

**main.py**

In [ ]:
# Save this code in main.py, in the same folder as calculator.py

import calculator

calc = calculator.Calculator()

print(calc.add(23, 34, 43))
print(calc.add(1, 2, 3, 4, 5))
print(calc.get_history())

This separates responsibilities:
- `calculator.py` contains the reusable calculator class.
- `main.py` decides how to use that class in this particular program.
- The history is stored on the calculator instance.

# Part 4 — What happens during import?

## 4.1 Top-level statements execute
When Python imports a module for the first time in a process, it executes the module's top-level code. Function and class definitions are created; top-level statements such as `print(...)` are also executed.

This explains why a `print()` placed directly in `calculator.py` appears when `main.py` imports `calculator`, even if `main.py` never explicitly calls that print statement.

In [ ]:
# Demonstration in a real file named noisy_module.py:
print("This message runs when noisy_module is imported")

def useful_function():
    return "Useful result"

If `main.py` contains `import noisy_module`, the top-level print executes during the first import. The function body does not execute until `useful_function()` is called.

## 4.2 `__name__` and the main guard
Python sets a special variable called `__name__` in every module:
- When a file is run directly, its `__name__` is `"__main__"`.
- When the file is imported as a module, `__name__` is usually the module's import name.

Use this condition to keep test/demo code from running automatically when another file imports the module.

In [ ]:
# Save as calculator.py

def add(a, b):
    return a + b

print("Module name:", __name__)

if __name__ == "__main__":
    print("Running calculator.py directly")
    print(add(10, 20))

Run it directly:

```bash
python calculator.py
```

Then import it from another file:

```python
import calculator
print(calculator.add(4, 5))
```

The function definition is available after import, but the code inside the `if __name__ == "__main__":` block does not run during that import.

## 4.3 Important technical clarification: `sys.path` and `sys.modules`
The transcript refers to a “system.path” mapping that stores imported function and class names. The actual Python concepts are different:

- `sys.path` is a list of locations Python searches when resolving imports.
- `sys.modules` is a dictionary/cache of modules already loaded in the current Python process.
- A module's functions and classes are attributes of that module object; they are not all stored as entries in `sys.path`.

Inspect them:

In [ ]:
import sys

print("First few import search locations:")
for location in sys.path[:5]:
    print(location)

print("Is math loaded?", "math" in sys.modules)
import math
print("Is math loaded now?", "math" in sys.modules)

The key idea from class is that Python resolves modules and keeps loaded modules available rather than repeatedly re-executing the same module on every ordinary import. The precise mechanisms are `sys.path` for search and `sys.modules` for the loaded-module cache.

# Part 5 — Packages

## 5.1 What is a package?
A **package** is a way to organize related modules in a directory structure. In the class example, `UDS3` is the package and files such as `healthcare.py`, `password.py` and `financial.py` are modules within it.

Example structure:

```text
project/
    main.py
    UDS3/
        __init__.py
        healthcare.py
        password.py
        financial.py
        sub_package/
            __init__.py
            c.py
```

The `__init__.py` file is used to initialize a package and can expose selected names at package level. Modern Python can also support namespace packages without `__init__.py`, but the class examples use regular packages with this file.

## 5.2 Importing a module from a package
If `main.py` is alongside the `UDS3` folder, you can import the `healthcare` module like this:

In [ ]:
# In main.py
from UDS3 import healthcare

daily_calories = healthcare.calculate_daily_calories(
    94, 190, 34, "male", "moderate"
)
print(daily_calories)

bmi = healthcare.calculate_bmi(94, 1.9)
print(bmi)

Here:
- `UDS3` is the package.
- `healthcare` is the module.
- `calculate_bmi` and `calculate_daily_calories` are functions inside that module.

## 5.3 Import a specific function from a module

In [ ]:
# In main.py
from UDS3.password import print_password

print_password("example-password")

This imports only the name `print_password` into the current file's namespace. The module is still imported and its top-level code is executed as part of the import; importing one name does not mean Python skips loading the module.

## 5.4 Expose selected names through `__init__.py`
The supplied `UDS3/__init__.py` contained:

```python
from .password import check_password_strength

WEEKEND_DAYS = ("SAT", "SUN")
```

The leading dot means “from this package/current package directory.” This makes `check_password_strength` and `WEEKEND_DAYS` available from the package level.

In [ ]:
# UDS3/__init__.py

from .password import check_password_strength

WEEKEND_DAYS = ("SAT", "SUN")

Now a consumer can write:

In [ ]:
# main.py
from UDS3 import check_password_strength, WEEKEND_DAYS

stats = check_password_strength("123#$%^ABC")
print(stats)
print(WEEKEND_DAYS)

Without that re-export in `__init__.py`, the direct import would instead be:

```python
from UDS3.password import check_password_strength
```

`__init__.py` therefore provides a package-level interface. Expose only names that you intentionally want consumers to import from the package root.

## 5.5 Importing multiple names

In [ ]:
from UDS3.healthcare import calculate_bmi, calculate_daily_calories

print(calculate_bmi(70, 1.75))
print(calculate_daily_calories(70, 175, 30, "female", "light"))

The class also showed `from UDS3.healthcare import *`. This imports names that the module makes available for star imports. It is convenient for demonstrations but is usually less clear than listing the names explicitly.

## 5.6 Nested packages and relative imports
The supplied structure included `UDS3/sub_package/c.py`. A function inside it was imported using the full dotted path:

In [ ]:
# In UDS3/sub_package/c.py

def add(a, b):
    print(a, b)

In [ ]:
# In main.py
from UDS3.sub_package.c import add

add(10, 20)

Each dot in the import path moves one level through the package/module hierarchy:
- `UDS3` — top-level package
- `sub_package` — subpackage
- `c` — module
- `add` — function inside the module

# Part 6 — Applied package example from the class

The supplied archive contains a `UDS3` package with healthcare, password and financial modules. The following summaries describe the actual functions in those files.

## 6.1 Healthcare module
`calculate_bmi(weight_kg, height_meters)`:
- Checks that weight and height are greater than zero.
- Calculates BMI and rounds it to one decimal place.
- Returns a dictionary containing the BMI and category.

`calculate_daily_calories(weight_kg, height_cm, age_years, gender, activity_level)`:
- Uses a gender-specific BMR formula.
- Applies an activity multiplier.
- Returns BMR calories and maintenance calories in a dictionary.
- Defaults to the sedentary multiplier when the activity label is not found.

In [ ]:
def calculate_bmi(weight_kg, height_meters):
    if height_meters <= 0 or weight_kg <= 0:
        return "Error: Weight and height must be greater than zero."

    bmi = round(weight_kg / (height_meters ** 2), 1)

    if bmi < 18.5:
        category = "Underweight"
    elif bmi < 25.0:
        category = "Normal weight"
    elif bmi < 30.0:
        category = "Overweight"
    else:
        category = "Obesity"

    return {"bmi": bmi, "category": category}

print(calculate_bmi(70, 1.75))

## 6.2 Password module
The supplied `password.py` checks five criteria: minimum length, uppercase, lowercase, digit and special character. It returns a dictionary with a strength label, score out of five and individual validation flags.

The supplied package initializer re-exports `check_password_strength` so callers can import it from `UDS3`.

In [ ]:
def check_password_strength(password):
    has_length = len(password) >= 8
    has_upper = any(char.isupper() for char in password)
    has_lower = any(char.islower() for char in password)
    has_digit = any(char.isdigit() for char in password)
    has_special = any(char in "!@#$%^&*(),.?\":{}|<>" for char in password)

    score = sum([has_length, has_upper, has_lower, has_digit, has_special])

    if score == 5:
        strength = "Strong"
    elif 3 <= score <= 4:
        strength = "Moderate"
    else:
        strength = "Weak"

    return {
        "strength": strength,
        "score_out_of_5": score,
        "validations": {
            "length_ok": has_length,
            "upper_case_ok": has_upper,
            "lower_case_ok": has_lower,
            "digit_ok": has_digit,
            "special_char_ok": has_special,
        },
    }

print(check_password_strength("123#$%^ABC"))

## 6.3 Financial module
The supplied `financial.py` defines `months_to_savings_goal(...)`. It estimates how many monthly periods are needed to reach a target, applying monthly interest and a monthly deposit. It includes a 600-month safety limit and returns either an error message or a dictionary with months, years and final balance.

This notebook summarizes the supplied module; it does not independently validate the financial assumptions or provide financial advice.

In [ ]:
def months_to_savings_goal(starting_balance, monthly_deposit, annual_interest_rate, target_goal):
    monthly_interest_rate = (annual_interest_rate / 100) / 12
    current_balance = starting_balance
    months = 0

    if current_balance < target_goal and monthly_deposit <= 0 and annual_interest_rate <= 0:
        return "Error: Goal is impossible to reach with zero growth."

    while current_balance < target_goal:
        current_balance += current_balance * monthly_interest_rate
        current_balance += monthly_deposit
        months += 1

        if months > 600:
            return "Goal takes over 50 years to reach. Consider increasing deposits."

    return {
        "months": months,
        "years": round(months / 12, 1),
        "final_balance": round(current_balance, 2),
    }

print(months_to_savings_goal(1000, 500, 6, 10000))

# Part 7 — Common mistakes and corrections

| Mistake | Why it is a problem | Better approach |
|---|---|---|
| Thinking `time.time()` returns milliseconds | It returns seconds since the Unix epoch | Label the unit as seconds |
| Using `perf_counter()` as a calendar timestamp | Its reference point is undefined | Subtract two readings for duration |
| Writing `wrapper()` with no parameters around a function that accepts arguments | Calls with arguments fail | Use `*args, **kwargs` and forward them |
| Forgetting `return func(...)` in a wrapper | The decorated function may lose its return value | Return the wrapped result |
| Placing test prints at module top level | They run during import | Put test/demo code under the main guard |
| Treating `sys.path` as a list of function names | It stores import search locations | Use `sys.path` for search locations and `sys.modules` for loaded modules |
| Importing `*` everywhere | Names become hard to trace and may clash | Prefer explicit imports |
| Exposing every package function through `__init__.py` | Package API becomes broad and ambiguous | Re-export only intentional public names |

# Part 8 — Dry-run and practice exercises

## 8.1 Predict before running
What is printed, and why?

In [ ]:
def outer(func):
    def inner(*args, **kwargs):
        print("A")
        result = func(*args, **kwargs)
        print("B")
        return result
    return inner

@outer
def greet(name):
    print("Hello", name)
    return name.upper()

answer = greet("Subhash")
print(answer)

**Expected flow:** `A` prints first, then the original `greet` runs and prints `Hello Subhash`, then `B` prints, then the returned uppercase value is printed. The final value is `SUBHASH`.

## 8.2 Practice tasks
1. Use `time.perf_counter()` to compare two different ways of summing the same list. Keep the input data identical.
2. Write a decorator that prints the function name before calling it and returns the function's result.
3. Write a decorator that accepts any positional and keyword arguments.
4. Stack two decorators: one strips a string and another converts it to lowercase. Predict the order before running.
5. Create `geometry.py` with `area_of_circle()` and `area_of_rectangle()`. Import only the function you need from `main.py`.
6. Create a package named `utilities` with `math_tools.py` and `text_tools.py`.
7. Use `utilities/__init__.py` to expose one selected function from each module.
8. Add a main guard to a module containing a small self-test. Verify that the self-test runs directly but not when imported.

## 8.3 Sample solution — simple reusable decorator

In [ ]:
from functools import wraps

def announce(func):
    @wraps(func)
    def wrapper(*args, **kwargs):
        print(f"Running {func.__name__}")
        result = func(*args, **kwargs)
        print("Done")
        return result
    return wrapper

@announce
def area_of_rectangle(length, width):
    return length * width

print(area_of_rectangle(5, 4))

# Part 9 — Quick revision sheet

| Topic | One-line memory |
|---|---|
| `time.time()` | Wall-clock timestamp in seconds since Unix epoch |
| `time.perf_counter()` | High-resolution elapsed-time measurement |
| `time.sleep(n)` | Pause current thread for approximately `n` seconds |
| Function object | Refer to a function by its name without calling it |
| `func()` | Call the function |
| Decorator | Takes a callable and returns an enhanced callable |
| Wrapper | Inner function that adds behaviour around the original |
| `*args` | Collect positional arguments into a tuple |
| `**kwargs` | Collect keyword arguments into a dictionary |
| `@decorator` | Shorthand for rebinding the function name to decorator output |
| Stacked decorators | Applied from bottom to top |
| Module | A `.py` file containing reusable code |
| Package | A directory organizing related modules |
| `__init__.py` | Package initialization and optional package-level exports |
| `from x import y` | Import a selected name |
| `sys.path` | Locations searched for imports |
| `sys.modules` | Cache/dictionary of loaded modules |
| `__name__` | Special module variable |
| Main guard | Run code only when the file is executed directly |

# Source coverage and editorial notes

## Files used
- `24-10-2026.ipynb` — 53-cell class notebook covering `time`, decorators, wrapper functions, decorator arguments, stacked decorators and string transformation decorators.
- `calculator.py` — calculator class, history tracking, `time.time()` and `__name__` main guard.
- `healthcare_test.py` — BMI and daily-calorie functions.
- `main.py`, `main_v2.py`, `main_v3.py` — module and package import examples.
- `UDS3/healthcare.py`, `UDS3/password.py`, `UDS3/financial.py` — package modules.
- `UDS3/__init__.py` — package-level export and `WEEKEND_DAYS`.
- `UDS3/sub_package/c.py` — nested package module.
- Class transcript for 24 October 2026.

## Technical clarifications
- `time.time()` returns seconds since the Unix epoch, not milliseconds. The instructor explicitly corrected this during the session.
- `time.perf_counter()` is presented as the preferred high-resolution duration timer; its absolute reference point is undefined.
- The transcript's “system.path” description is corrected to the actual Python mechanisms: `sys.path` (import search locations) and `sys.modules` (loaded module cache).
- Decorator wrappers are shown forwarding arguments and returning results, making the examples reusable and preserving function behaviour.
- The supplied calculator class name `Calulcator` is corrected to `Calculator` in the teaching version.
- The PDF was image-based or otherwise had no extractable text layer, so no unsupported PDF content has been added.

## Scope boundary
The transcript introduces modules and packages after decorators and discusses that class methods and other topics would be covered later. This notebook does not claim those later topics were taught in this session.